# 네팔 건물 피해 등급 분류 실험

전체 ZIP을 압축 해제한 폴더에서 이 노트북을 여세요. 동봉된 학습 모델로 결과를 확인하고 새 건물을 예측할 수 있습니다. 아래 출력은 첨부 CSV를 이용한 실제 실행 기록에서 가져옵니다.

**주지표: Macro F1.** 테스트는 모델 선택과 예측 규칙 조정에 사용하지 않았습니다. 같은 지진·같은 지역 분포 안의 건물 단위 무작위 평가입니다.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib

PROJECT = Path.cwd()
EXP = PROJECT / 'nepal_experiments'
if not EXP.exists():
    raise FileNotFoundError('전체 ZIP을 압축 해제한 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(EXP))
from nepal_core import load_data
from finalize_model import combine
from sklearn.metrics import f1_score, classification_report
print('실험 폴더:', EXP.name)

실험 폴더: nepal_experiments


## 1 데이터와 평가 분할

building_id는 입력에서 제외합니다. 층화 분할을 70% / 15% / 15%로 고정했습니다. 최종 모델은 학습+검증 85%로 재학습했습니다.

In [2]:
audit = json.loads((EXP / 'data_audit.json').read_text())
print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "rows": 260601,
  "columns": 40,
  "source_sha256": "87abd2bc2b05d3e687bd176959967fd38334fff607e6afeb2d20d0d899c13e97",
  "split": {
    "train": 182420,
    "validation": 39090,
    "test": 39091
  },
  "class_counts": {
    "1": 25124,
    "2": 148259,
    "3": 87218
  },
  "missing": 0,
  "building_id_unique": true,
  "duplicate_feature_rows": 16191,
  "age_995": 1390,
  "geo_level_1_id": {
    "unique": 31,
    "validation_unseen_fraction": 0.0
  },
  "geo_level_2_id": {
    "unique": 1414,
    "validation_unseen_fraction": 0.00023023791250959325
  },
  "geo_level_3_id": {
    "unique": 11595,
    "validation_unseen_fraction": 0.00636991557943208
  }
}


## 2 전체 실험 비교

macro_f1은 기본 argmax 결정, tuned_macro_f1은 검증에서 클래스별 점수 배수를 조정한 결과입니다. 검증 최고 결과에는 선택 편향이 있을 수 있으므로 마지막 테스트 결과와 구분합니다.

In [3]:
experiments = pd.read_csv(EXP / 'experiment_results.csv')
print(experiments[['name','features','macro_f1','tuned_macro_f1','micro_f1']].to_string(index=False))

                  name  features  macro_f1  tuned_macro_f1  micro_f1
        hgb_rich_macro       118  0.707178        0.710539  0.747889
           hgb_hier_31       143  0.702346        0.710533  0.749706
       hgb_rich_10fold       118  0.701729        0.710189  0.749936
               hgb_ovr       118  0.703532        0.710143  0.750780
           hgb_hier_s5       143  0.702024        0.710010  0.750704
           hgb_hier_63       143  0.701053        0.709934  0.749808
      hgb_neighbors_63       165  0.702962        0.709876  0.750524
            hgb_eng_s5        93  0.700524        0.709839  0.748785
       hgb_hier_10fold       143  0.702051        0.709791  0.750652
           hgb_rich_63       118  0.704150        0.709611  0.751317
         hgb_neighbors       165  0.704669        0.709485  0.751906
     hgb_rich_seed2026       118  0.704612        0.709447  0.751343
      hgb_eng_weighted        93  0.708311        0.709344  0.746917
          hgb_rich_255       118  

## 3 피처 엔지니어링

- 지역별 클래스 비율과 빈도, 지역 평균 대비 건물 특성의 차이
- 층당 높이, 가족당 면적, 면적×층수 등의 규모 지표
- 재료 개수·조합, 기초×지붕 등 범주 조합
- 지역×재료·기초·지붕 교차 인코딩과 상위 지역으로 평활화
- 동일 지역의 유사 건물 3·10·30개에 대한 피해 경향
- 타깃 정보는 학습 내부 5-fold 또는 10-fold에서 자기 정답을 제외하고 생성

세부 구현은 nepal_core.py와 local_neighbors.py에 있습니다. 모든 방법이 점수를 높이지는 않았습니다.

In [4]:
selection = json.loads((EXP / 'selection.json').read_text())
print(json.dumps(selection, ensure_ascii=False, indent=2))

{
  "members": [
    "hgb_hier_31",
    "hgb_eng_weighted",
    "rf_raw",
    "hgb_rich_63"
  ],
  "member_multipliers": [
    [
      1.480845958362467,
      1.0,
      1.199126800203751
    ],
    [
      1.0328856651449307,
      1.0,
      1.0742220776634044
    ],
    [
      1.893776761368618,
      1.0,
      1.2558197020661348
    ],
    [
      1.3708275651492945,
      1.0,
      1.171218673149492
    ]
  ],
  "blend_weights": [
    0.16495724371848783,
    0.25861863989896094,
    0.22778697048990362,
    0.34863714589264755
  ],
  "decision_multipliers": [
    1.248056443405485,
    1.0,
    0.8692713673800894
  ],
  "validation_macro_f1": 0.7141704673704573,
  "selection_basis": "Validation Macro F1; final test not used for selection.",
  "multiplicative_validation_macro_f1": 0.7141704673704573,
  "decision_biases": [
    0.0,
    0.0,
    0.0
  ],
  "data_sha256": "87abd2bc2b05d3e687bd176959967fd38334fff607e6afeb2d20d0d899c13e97"
}


## 4 독립 테스트 결과

최종 구성과 결정 규칙을 고정한 뒤 평가한 결과입니다. 이 값을 보고 모델을 다시 선택하지 않았습니다. baseline_hgb_70pct는 동일 테스트의 기본 HGB, selected_70pct는 재학습 전 선택한 앙상블입니다.

In [5]:
results = json.loads((EXP / 'final_test_metrics.json').read_text())
for name in ['baseline_hgb_70pct','selected_70pct','final_refit_85pct']:
    r = results[name]
    print(name, 'Macro F1=', round(r['macro_f1'], 6), 'Accuracy=', round(r['accuracy'], 6))
print('등급별 지표')
print(pd.DataFrame(results['final_refit_85pct']['classification_report']).T.to_string())

baseline_hgb_70pct Macro F1= 0.691801 Accuracy= 0.745415
selected_70pct Macro F1= 0.71821 Accuracy= 0.751708
final_refit_85pct Macro F1= 0.720166 Accuracy= 0.752884
등급별 지표
              precision    recall  f1-score       support
1              0.599123  0.724861  0.656021   3769.000000
2              0.774075  0.811457  0.792325  22239.000000
3              0.771350  0.661393  0.712152  13083.000000
accuracy       0.752884  0.752884  0.752884      0.752884
macro avg      0.714849  0.732570  0.720166  39091.000000
weighted avg   0.756294  0.752884  0.752351  39091.000000


## 5 저장된 테스트 예측으로 점수 재계산

이 셀은 학습 없이 결과 CSV의 정답과 예측을 이용해 평가 점수를 확인합니다.

In [6]:
predictions = pd.read_csv(EXP / 'test_predictions.csv')
score = f1_score(predictions.actual_damage_grade, predictions.predicted_damage_grade, average='macro')
print(f'재계산한 테스트 Macro F1: {score:.8f}')
assert np.isclose(score, results['final_refit_85pct']['macro_f1'])

재계산한 테스트 Macro F1: 0.72016605


## 6 학습된 모델로 직접 예측

처음에는 테스트의 20건으로 저장된 예측과 일치하는지 확인합니다. sample을 동일 컬럼의 새 CSV로 교체하면 새로운 건물도 예측할 수 있습니다. score_grade는 결정용 점수이며 실제 피해 확률로 보정된 값은 아닙니다. 모델 파일은 직접 제공받은 신뢰할 수 있는 파일만 불러오세요.

In [7]:
RUN_SAVED_MODEL = True
if RUN_SAVED_MODEL:
    bundle = joblib.load(EXP / 'final_model.joblib')
    raw, y, train_idx, valid_idx, test_idx = load_data()
    sample = raw.iloc[test_idx[:20]]
    scores = combine(bundle['members'], sample, bundle['selection'])
    labels = scores.argmax(axis=1) + 1
    assert np.array_equal(labels, predictions.predicted_damage_grade.iloc[:20].to_numpy())
    print(pd.DataFrame({'building_id': sample.building_id.to_numpy(), '정답': sample.damage_grade.to_numpy(), '예측': labels}).to_string(index=False))

 building_id  정답  예측
      110972   2   2
       83861   3   3
      701146   3   3
      987778   3   2
      956109   2   2
      318312   3   3
      425664   2   2
     1049568   2   2
      408815   2   2
      484176   2   1
      743570   2   2
      295535   1   2
      205142   2   3
      599842   3   3
      672752   3   2
      842891   2   1
      288585   2   3
      600888   2   2
      727042   3   2
      550492   2   2


## 7 전체 탐색 다시 실행

아래 스위치는 기본 False입니다. True로 바꾸면 빠진 실험 모델을 새로 학습합니다. 컴퓨터에 따라 수십 분 이상 걸리고, 중간 모델·캐시에 수 GB가 필요할 수 있습니다. 학습 CPU 스레드는 NEPAL_THREADS와 OMP_NUM_THREADS로 조절할 수 있습니다.

기존 final_test_metrics.json이 있으면 finalize_model.py는 재평가를 건너뜁니다. 같은 테스트를 계속 보면서 설정을 바꾸지 않도록 한 동작입니다. 원본 기록을 보존한 별도 작업 복사본에서만 전체 재실험을 진행하세요.

In [8]:
RUN_FULL_SEARCH = False
if RUN_FULL_SEARCH:
    import subprocess
    for stage in ['one','two','baseline','three','four','five','six']:
        subprocess.run([sys.executable, str(EXP / 'run_experiments.py'), '--stage', stage], check=True)
    subprocess.run([sys.executable, str(EXP / 'select_ensemble.py')], check=True)
    subprocess.run([sys.executable, str(EXP / 'finalize_model.py')], check=True)
else:
    print('저장된 실행 결과를 사용했습니다. 전체 재학습은 RUN_FULL_SEARCH=True로 실행합니다.')

저장된 실행 결과를 사용했습니다. 전체 재학습은 RUN_FULL_SEARCH=True로 실행합니다.


## 8 한계

지역 단위 분리 검증과 다른 지진으로의 외부 검증은 하지 않았습니다. 95% 구간은 고정 모델의 테스트 행을 재표집한 값이며 탐색 불확실성은 포함하지 않습니다. 로지스틱 회귀는 300회 반복에서 수렴 경고가 있었습니다. 최종 결과는 실행한 후보 중 검증 기준으로 선택한 결과이며 절대적인 최고 성능을 보장하지 않습니다.